# Snowflake Time Travel & Fail-safe Tutorial

This notebook demonstrates **Time Travel** — Snowflake's ability to query, clone, and restore historical data.

**What you'll learn:**
1. Query data at a point in time using `AT` and `BEFORE`
2. Recover from accidental DML (UPDATE, DELETE, DROP)
3. Clone tables from a historical point
4. Understand data retention and Fail-safe

**Pattern:** Setup → Demo → Cleanup

## 1. Setup — Create Database, Schema & Table

In [ ]:
%%sql -r setup_result
CREATE OR REPLACE DATABASE time_travel_tutorial_db;
CREATE SCHEMA time_travel_tutorial_db.tutorial;

In [ ]:
%%sql -r table_result
CREATE OR REPLACE TABLE time_travel_tutorial_db.tutorial.employees (
    employee_id INT,
    employee_name VARCHAR(100),
    department VARCHAR(50),
    salary DECIMAL(10,2),
    hire_date DATE
)
DATA_RETENTION_TIME_IN_DAYS = 1;

## 2. Insert Initial Data

In [ ]:
%%sql -r insert_result
INSERT INTO time_travel_tutorial_db.tutorial.employees VALUES
    (1, 'Alice Johnson', 'Engineering', 95000.00, '2022-03-15'),
    (2, 'Bob Smith', 'Marketing', 72000.00, '2021-07-01'),
    (3, 'Charlie Brown', 'Engineering', 88000.00, '2023-01-10'),
    (4, 'Diana Ross', 'Finance', 91000.00, '2020-11-20'),
    (5, 'Edward King', 'Marketing', 68000.00, '2024-02-28');

In [ ]:
%%sql -r initial_data
SELECT * FROM time_travel_tutorial_db.tutorial.employees ORDER BY employee_id;

## 3. Capture a Timestamp Before Making Changes

We record the current timestamp so we can travel back to this exact point.

In [ ]:
%%sql -r snapshot_time
SELECT CURRENT_TIMESTAMP() AS snapshot_time;

## 4. Make Accidental Changes

In [ ]:
%%sql -r bad_update
UPDATE time_travel_tutorial_db.tutorial.employees
SET salary = 50000.00
WHERE department = 'Engineering';

In [ ]:
%%sql -r bad_delete
DELETE FROM time_travel_tutorial_db.tutorial.employees
WHERE department = 'Marketing';

In [ ]:
%%sql -r after_damage
SELECT * FROM time_travel_tutorial_db.tutorial.employees ORDER BY employee_id;

## 5. Time Travel — Query Historical Data

Use `AT(TIMESTAMP => ...)` to see the data as it was before our mistakes.

In [ ]:
%%sql -r time_travel_offset
SELECT * FROM time_travel_tutorial_db.tutorial.employees
    AT(OFFSET => -120)
ORDER BY employee_id;

In [ ]:
%%sql -r time_travel_stmt
SELECT
    employee_id,
    employee_name,
    department,
    salary
FROM time_travel_tutorial_db.tutorial.employees
    BEFORE(STATEMENT => LAST_QUERY_ID(-2))
ORDER BY employee_id;

## 6. Restore Data Using Time Travel

Recreate the table from its historical state using `CREATE TABLE ... CLONE ... AT`.

In [ ]:
%%sql -r clone_result
CREATE OR REPLACE TABLE time_travel_tutorial_db.tutorial.employees_restored
    CLONE time_travel_tutorial_db.tutorial.employees
    AT(OFFSET => -120);

In [ ]:
%%sql -r restored_data
SELECT * FROM time_travel_tutorial_db.tutorial.employees_restored ORDER BY employee_id;

## 7. UNDROP — Recover a Dropped Table

In [ ]:
%%sql -r drop_result
DROP TABLE time_travel_tutorial_db.tutorial.employees;

In [ ]:
%%sql -r undrop_result
UNDROP TABLE time_travel_tutorial_db.tutorial.employees;

In [ ]:
%%sql -r undrop_verify
SELECT COUNT(*) AS row_count FROM time_travel_tutorial_db.tutorial.employees;

## 8. Check Data Retention Settings

In [ ]:
%%sql -r retention_info
SHOW TABLES LIKE 'employees%' IN SCHEMA time_travel_tutorial_db.tutorial;

In [ ]:
%%sql -r alter_retention
ALTER TABLE time_travel_tutorial_db.tutorial.employees
    SET DATA_RETENTION_TIME_IN_DAYS = 3;

## 9. Cleanup — Drop All Tutorial Objects

In [ ]:
%%sql -r cleanup_result
DROP DATABASE IF EXISTS time_travel_tutorial_db;